# Predicción fenológica reproducible en Google Colab

Este notebook reproduce el flujo técnico desde los CSV consolidados: valida integridad, prepara A/A′/B, compara red densa y Random Forest, entrena Random Forest A y ejecuta una inferencia. Los datos permanecen en Google Drive privado.

**Fuente congelada:** `phenological_prediction` 0.1.0-rc.2, commit `96b4e94687e8ff0aa7f904509ec0c2cdb4f0751d`.

> Alcance académico: datos europeos, sin validación operacional en Chile y sin sustitución del criterio agronómico.

## 1. Dependencias
Las versiones corresponden al RC verificado. La instalación se realiza antes de importar las bibliotecas de modelado.

In [ ]:
import subprocess
import sys

PAQUETES = [
    'tensorflow==2.21.0',
    'pandas==2.3.3',
    'scikit-learn==1.6.1',
    'matplotlib==3.10.9',
    'numpy==2.5.2',
    'joblib==1.5.3',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *PAQUETES])
print('Dependencias instaladas. Python:', sys.version)

## 2. Configuración privada de Drive
La cuenta que autorice el montaje debe tener acceso a la carpeta. Los CSV no se copian al notebook ni al repositorio.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import hashlib
import json
import os
import platform
import random
import shutil
from datetime import datetime
from pathlib import Path

# Modificar solamente si se eligió otra carpeta privada en MyDrive.
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/phenological_prediction_private')
DRIVE_DATA_DIR = DRIVE_PROJECT_ROOT / 'data'
DRIVE_RUNS_DIR = DRIVE_PROJECT_ROOT / 'ejecuciones'

EJECUTAR_COMPARACION_COMPLETA = True
VALORES_DEMO = [18.5, 24.0, 12.0, 15.0, 18.2, 62.0, 210.0]

RUN_ID = datetime.now().astimezone().strftime('%Y%m%d_%H%M%S_%f')
RUN_DIR_DRIVE = DRIVE_RUNS_DIR / RUN_ID
RUN_DIR_LOCAL = Path('/content') / f'phenological_prediction_run_{RUN_ID}'
DATA_DIR_LOCAL = RUN_DIR_LOCAL / 'data'
OUTPUT_DIR_LOCAL = RUN_DIR_LOCAL / 'output'
MODELS_DIR_LOCAL = RUN_DIR_LOCAL / 'models'
for carpeta in (DRIVE_RUNS_DIR, DATA_DIR_LOCAL, OUTPUT_DIR_LOCAL, MODELS_DIR_LOCAL):
    carpeta.mkdir(parents=True, exist_ok=True)

HASHES_DATOS = {
    'base_fenologia_clima.csv': '0397C7A0B61B76388C22A1CDD1F13BCB2B7E10069C7BBB2935F0ADCC2E5CF6B7',
    'base_fenologia_clima_satelite.csv': '0C307E8CAAEEE04A87EB572AA675C4BB7C97EF1C1BC5C0D36C7018FB7129ADCA',
}

def sha256_texto_lf(ruta: Path) -> str:
    contenido = ruta.read_bytes().replace(b'\r\n', b'\n')
    return hashlib.sha256(contenido).hexdigest().upper()

for nombre, esperado in HASHES_DATOS.items():
    origen = DRIVE_DATA_DIR / nombre
    if not origen.exists():
        raise FileNotFoundError(f'Falta en Google Drive: {origen}')
    observado = sha256_texto_lf(origen)
    if observado != esperado:
        raise ValueError(f'Hash incorrecto para {nombre}: {observado}; esperado: {esperado}')
    shutil.copy2(origen, DATA_DIR_LOCAL / nombre)
    print(f'OK {nombre}: {observado}')

print('Datos verificados y copiados al almacenamiento temporal de Colab.')

## 3. Parámetros y funciones del flujo
Todas las variantes comparten semilla, folds, definición de variables y parámetros. La imputación se ajusta exclusivamente con entrenamiento dentro de cada fold.

In [ ]:
os.environ['PYTHONHASHSEED'] = '42'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import tensorflow as tf
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

CLIMA = [
    'clima_temp_media', 'clima_temp_max_media', 'clima_temp_min_media',
    'clima_precip_acumulada', 'clima_radiacion_media',
    'clima_humedad_media', 'clima_gdd_acumulado',
]
CONFIGURACION = {
    'version_notebook': '0.1.0-notebook.1',
    'fuente_rc': '0.1.0-rc.2',
    'commit_fuente': '96b4e94687e8ff0aa7f904509ec0c2cdb4f0751d',
    'semilla': 42,
    'folds': 5,
    'random_forest': {
        'n_estimators': 400, 'class_weight': 'balanced',
        'random_state': 42, 'n_jobs': -1,
    },
    'modelos': {
        'a': ('base_fenologia_clima.csv', CLIMA, 'Modelo A - clima'),
        'aprima': ('base_fenologia_clima_satelite.csv', CLIMA, "Modelo A' - control clima"),
        'b': ('base_fenologia_clima_satelite.csv', CLIMA + ['ndvi'], 'Modelo B - clima + NDVI'),
    },
}

def fijar_semillas() -> None:
    semilla = CONFIGURACION['semilla']
    random.seed(semilla)
    np.random.seed(semilla)
    tf.keras.utils.set_random_seed(semilla)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception as error:
        print('Aviso: no se pudo activar determinismo completo:', error)

def cargar_datos(nombre: str):
    archivo, variables, titulo = CONFIGURACION['modelos'][nombre]
    datos = pd.read_csv(DATA_DIR_LOCAL / archivo)
    if nombre in {'aprima', 'b'}:
        datos = datos[datos['ndvi'].notna()].copy()
    if nombre == 'a':
        requeridas = [v for v in variables if v != 'clima_radiacion_media'] + ['macro_etapa', 's_id']
    else:
        requeridas = variables + ['macro_etapa', 's_id']
    datos = datos.dropna(subset=requeridas).reset_index(drop=True)
    return datos, variables, titulo

def crear_red(n_variables: int, n_clases: int):
    red = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(n_variables,)),
        tf.keras.layers.Dense(16, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(8, activation='relu'),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(n_clases, activation='softmax'),
    ])
    red.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy', metrics=['accuracy'],
    )
    return red

def crear_particiones(x, y, grupos, validacion):
    if validacion == 'aleatoria':
        divisor = StratifiedKFold(
            n_splits=CONFIGURACION['folds'], shuffle=True,
            random_state=CONFIGURACION['semilla'],
        )
        return divisor.split(x, y)
    divisor = StratifiedGroupKFold(
        n_splits=CONFIGURACION['folds'], shuffle=True,
        random_state=CONFIGURACION['semilla'],
    )
    return divisor.split(x, y, groups=grupos)

def predecir_red(x_train, y_train, x_test, n_clases):
    escalador = StandardScaler()
    x_train = escalador.fit_transform(x_train)
    x_test = escalador.transform(x_test)
    clases_train = np.unique(y_train)
    pesos = compute_class_weight('balanced', classes=clases_train, y=y_train)
    red = crear_red(x_train.shape[1], n_clases)
    parada = tf.keras.callbacks.EarlyStopping(
        monitor='loss', patience=8, restore_best_weights=True,
    )
    red.fit(
        x_train, y_train, epochs=60, batch_size=16, verbose=0,
        class_weight=dict(zip(clases_train, pesos)), callbacks=[parada],
    )
    prediccion = np.argmax(red.predict(x_test, verbose=0), axis=1)
    tf.keras.backend.clear_session()
    return prediccion

def evaluar(nombre: str, validacion: str):
    datos, variables, titulo = cargar_datos(nombre)
    codificador = LabelEncoder()
    y = codificador.fit_transform(datos['macro_etapa'])
    x = datos[variables].to_numpy()
    grupos = datos['s_id'].to_numpy()
    n_clases = len(codificador.classes_)
    matrices = {
        'red_densa': np.zeros((n_clases, n_clases), dtype=int),
        'random_forest': np.zeros((n_clases, n_clases), dtype=int),
    }
    filas = []
    for fold, (idx_train, idx_test) in enumerate(
        crear_particiones(x, y, grupos, validacion), start=1
    ):
        estaciones_train = set(grupos[idx_train])
        estaciones_test = set(grupos[idx_test])
        compartidas = estaciones_train.intersection(estaciones_test)
        if validacion == 'por_estacion' and compartidas:
            raise ValueError(f'Estaciones compartidas en {nombre}, fold {fold}: {compartidas}')
        imputador = SimpleImputer(strategy='median')
        x_train = imputador.fit_transform(x[idx_train])
        x_test = imputador.transform(x[idx_test])
        bosque = RandomForestClassifier(**CONFIGURACION['random_forest'])
        predicciones = {
            'red_densa': predecir_red(x_train, y[idx_train], x_test, n_clases),
            'random_forest': bosque.fit(x_train, y[idx_train]).predict(x_test),
        }
        etiquetas = list(range(n_clases))
        for clasificador, prediccion in predicciones.items():
            matrices[clasificador] += confusion_matrix(
                y[idx_test], prediccion, labels=etiquetas
            )
            filas.append({
                'modelo_datos': nombre, 'validacion': validacion,
                'clasificador': clasificador, 'fold': fold,
                'n_train': len(idx_train), 'n_test': len(idx_test),
                'estaciones_train': len(estaciones_train),
                'estaciones_test': len(estaciones_test),
                'estaciones_compartidas': len(compartidas),
                'accuracy': accuracy_score(y[idx_test], prediccion),
                'f1_macro': f1_score(
                    y[idx_test], prediccion, labels=etiquetas,
                    average='macro', zero_division=0,
                ),
                'f1_weighted': f1_score(
                    y[idx_test], prediccion, labels=etiquetas,
                    average='weighted', zero_division=0,
                ),
            })
    for clasificador, matriz in matrices.items():
        tabla = pd.DataFrame(
            matriz, index=codificador.classes_, columns=codificador.classes_
        )
        tabla.index.name = 'clase_real'
        tabla.to_csv(OUTPUT_DIR_LOCAL / f'matriz_{nombre}_{validacion}_{clasificador}.csv')
    metadata = {
        'modelo_datos': nombre, 'titulo': titulo, 'validacion': validacion,
        'muestras': int(len(datos)), 'estaciones': int(datos['s_id'].nunique()),
        'variables': variables, 'clases': list(codificador.classes_),
    }
    return pd.DataFrame(filas), metadata

fijar_semillas()
for nombre in CONFIGURACION['modelos']:
    datos, variables, titulo = cargar_datos(nombre)
    print(f'{titulo}: {len(datos)} muestras, {datos.s_id.nunique()} estaciones, {len(variables)} variables')

## 4. Comparación completa
Se generan 60 registros: 3 conjuntos × 2 validaciones × 2 clasificadores × 5 folds. La validación por estación es la evidencia principal.

In [ ]:
if EJECUTAR_COMPARACION_COMPLETA:
    resultados = []
    metadatos = []
    for nombre in CONFIGURACION['modelos']:
        for validacion in ('aleatoria', 'por_estacion'):
            print(f'Ejecutando {nombre}: {validacion}')
            filas, metadata = evaluar(nombre, validacion)
            resultados.append(filas)
            metadatos.append(metadata)
    detalle = pd.concat(resultados, ignore_index=True)
    resumen = (
        detalle.groupby(['modelo_datos', 'validacion', 'clasificador'], as_index=False)
        .agg(
            accuracy_promedio=('accuracy', 'mean'),
            accuracy_desviacion=('accuracy', 'std'),
            f1_macro_promedio=('f1_macro', 'mean'),
            f1_macro_desviacion=('f1_macro', 'std'),
            f1_weighted_promedio=('f1_weighted', 'mean'),
        )
    )
    detalle.to_csv(OUTPUT_DIR_LOCAL / 'metricas_por_fold.csv', index=False)
    resumen.to_csv(OUTPUT_DIR_LOCAL / 'comparacion_consolidada.csv', index=False)
    etiquetas_grafico = [
        f'{fila.modelo_datos}\n{fila.validacion}\n{fila.clasificador}'
        for fila in resumen.itertuples()
    ]
    posiciones = np.arange(len(resumen))
    figura, eje = plt.subplots(figsize=(15, 6))
    eje.bar(posiciones - 0.18, resumen['accuracy_promedio'], 0.36, label='Accuracy')
    eje.bar(posiciones + 0.18, resumen['f1_macro_promedio'], 0.36, label='F1 macro')
    eje.set(
        title='Comparación de modelos y estrategias de validación',
        ylabel='Métrica promedio', ylim=(0, 1), xticks=posiciones,
        xticklabels=etiquetas_grafico,
    )
    plt.setp(eje.get_xticklabels(), rotation=45, ha='right')
    eje.grid(axis='y', alpha=0.3)
    eje.legend()
    figura.tight_layout()
    figura.savefig(OUTPUT_DIR_LOCAL / 'comparacion_metricas.png', dpi=160)
    plt.show()
    configuracion_salida = {
        **CONFIGURACION,
        'fecha_ejecucion': datetime.now().astimezone().isoformat(timespec='seconds'),
        'runtime': {
            'python': platform.python_version(), 'numpy': np.__version__,
            'pandas': pd.__version__, 'scikit_learn': sklearn.__version__,
            'tensorflow': tf.__version__, 'joblib': joblib.__version__,
        },
        'hashes_datos_sha256_lf': HASHES_DATOS,
        'metadatos_modelos': metadatos,
    }
    (OUTPUT_DIR_LOCAL / 'configuracion_ejecucion.json').write_text(
        json.dumps(configuracion_salida, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )
    referencia = {
        'accuracy_promedio': 0.8346299027206735,
        'accuracy_desviacion': 0.13726734207631164,
        'f1_macro_promedio': 0.7204350871824743,
        'f1_macro_desviacion': 0.11423188916818808,
    }
    observado = resumen.query(
        "modelo_datos == 'a' and validacion == 'por_estacion' and clasificador == 'random_forest'"
    ).iloc[0]
    verificacion = pd.DataFrame([
        {'metrica': metrica, 'referencia_rc': valor,
         'valor_colab': float(observado[metrica]),
         'diferencia': float(observado[metrica]) - valor}
        for metrica, valor in referencia.items()
    ])
    verificacion.to_csv(OUTPUT_DIR_LOCAL / 'verificacion_contra_rc.csv', index=False)
    display(resumen)
    display(verificacion)
else:
    resumen = None
    detalle = None
    print('Comparación omitida: este modo sirve para ensayo, no para reproducir métricas.')

## 5. Entrenamiento final de Random Forest A
El desempeño no se calcula con estas mismas 1.091 filas. Las métricas provienen de la validación agrupada anterior.

In [ ]:
datos_a, variables_a, _ = cargar_datos('a')
if len(datos_a) != 1091 or datos_a['s_id'].nunique() != 41:
    raise ValueError('Modelo A debe contener 1.091 muestras y 41 estaciones.')
modelo_final = Pipeline([
    ('imputador', SimpleImputer(strategy='median')),
    ('clasificador', RandomForestClassifier(**CONFIGURACION['random_forest'])),
])
modelo_final.fit(datos_a[variables_a], datos_a['macro_etapa'].astype(str))
rangos = {
    variable: {'min': float(datos_a[variable].min()), 'max': float(datos_a[variable].max())}
    for variable in variables_a
}
paquete = {
    'version_esquema': 1, 'pipeline': modelo_final,
    'variables': variables_a,
    'clases': list(modelo_final.named_steps['clasificador'].classes_),
    'rangos_entrenamiento': rangos,
    'advertencia': (
        'Uso experimental con datos europeos; no validado para operación en Chile '
        'y no sustituye evaluación agronómica.'
    ),
}
ruta_modelo = MODELS_DIR_LOCAL / 'random_forest_a_colab.joblib'
joblib.dump(paquete, ruta_modelo, compress=3)
hash_modelo = hashlib.sha256(ruta_modelo.read_bytes()).hexdigest().upper()
metadata_final = {
    'version_notebook': CONFIGURACION['version_notebook'],
    'fuente_rc': CONFIGURACION['fuente_rc'],
    'commit_fuente': CONFIGURACION['commit_fuente'],
    'fecha_entrenamiento': datetime.now().astimezone().isoformat(timespec='seconds'),
    'modelo': 'Random Forest A - clima', 'muestras_entrenamiento': 1091,
    'estaciones': 41, 'variables': variables_a,
    'clases': paquete['clases'],
    'parametros_random_forest': CONFIGURACION['random_forest'],
    'sha256_datos_lf': HASHES_DATOS['base_fenologia_clima.csv'],
    'sha256_modelo_colab': hash_modelo,
    'nota_hash': 'El hash serializado puede depender del entorno; no sustituye el modelo congelado del RC.',
    'advertencia': paquete['advertencia'],
}
(OUTPUT_DIR_LOCAL / 'metadata_modelo_colab.json').write_text(
    json.dumps(metadata_final, ensure_ascii=False, indent=2), encoding='utf-8'
)
print('Modelo Colab generado:', ruta_modelo)
print('SHA-256 del modelo Colab:', hash_modelo)

## 6. Inferencia demostrativa
Las probabilidades son la distribución relativa de votos del bosque y no están calibradas como certeza estadística.

In [ ]:
def ejecutar_demo(paquete_modelo: dict, valores: list[float]) -> dict:
    if len(valores) != len(paquete_modelo['variables']):
        raise ValueError('La demo requiere exactamente siete valores.')
    entrada = pd.DataFrame([valores], columns=paquete_modelo['variables'])
    modelo = paquete_modelo['pipeline']
    clase = str(modelo.predict(entrada)[0])
    probabilidades = modelo.predict_proba(entrada)[0]
    clases = list(modelo.named_steps['clasificador'].classes_)
    resultado = {
        'entrada': dict(zip(paquete_modelo['variables'], valores)),
        'macro_etapa_estimada': clase,
        'probabilidades_no_calibradas': {
            etiqueta: float(probabilidad)
            for etiqueta, probabilidad in zip(clases, probabilidades)
        },
        'advertencia': paquete_modelo['advertencia'],
    }
    print('Macro-etapa estimada:', clase)
    print('Probabilidades estimadas (no calibradas):')
    for etiqueta, probabilidad in sorted(
        resultado['probabilidades_no_calibradas'].items(),
        key=lambda elemento: elemento[1], reverse=True,
    ):
        print(f'  - {etiqueta}: {probabilidad:.1%}')
    print('Advertencia:', paquete_modelo['advertencia'])
    return resultado

resultado_demo = ejecutar_demo(paquete, VALORES_DEMO)
(OUTPUT_DIR_LOCAL / 'resultado_demo.json').write_text(
    json.dumps(resultado_demo, ensure_ascii=False, indent=2), encoding='utf-8'
)

## 7. Controles y exportación privada
La ejecución se acepta solo si los datos mantienen sus hashes, no existen estaciones compartidas en la validación agrupada y los resultados requeridos están completos.

In [ ]:
for nombre, esperado in HASHES_DATOS.items():
    assert sha256_texto_lf(DATA_DIR_LOCAL / nombre) == esperado
if EJECUTAR_COMPARACION_COMPLETA:
    assert len(detalle) == 60, f'Se esperaban 60 registros por fold; se obtuvieron {len(detalle)}.'
    assert len(resumen) == 12, f'Se esperaban 12 filas consolidadas; se obtuvieron {len(resumen)}.'
    agrupada = detalle[detalle['validacion'] == 'por_estacion']
    assert (agrupada['estaciones_compartidas'] == 0).all()
    assert not detalle[['accuracy', 'f1_macro', 'f1_weighted']].isna().any().any()

archivos_exportar = [
    *OUTPUT_DIR_LOCAL.glob('*'),
    *MODELS_DIR_LOCAL.glob('*'),
]
manifiesto = {
    'run_id': RUN_ID,
    'fecha_exportacion': datetime.now().astimezone().isoformat(timespec='seconds'),
    'fuente_rc': CONFIGURACION['fuente_rc'],
    'commit_fuente': CONFIGURACION['commit_fuente'],
    'comparacion_completa': EJECUTAR_COMPARACION_COMPLETA,
    'archivos': [],
}
for ruta in archivos_exportar:
    manifiesto['archivos'].append({
        'nombre': ruta.name,
        'sha256_raw': hashlib.sha256(ruta.read_bytes()).hexdigest().upper(),
    })
(RUN_DIR_LOCAL / 'manifest_ejecucion.json').write_text(
    json.dumps(manifiesto, ensure_ascii=False, indent=2), encoding='utf-8'
)

if RUN_DIR_DRIVE.exists():
    raise FileExistsError(f'La carpeta de ejecución ya existe: {RUN_DIR_DRIVE}')
RUN_DIR_DRIVE.mkdir(parents=True)
for subcarpeta in ('output', 'models'):
    origen = RUN_DIR_LOCAL / subcarpeta
    if origen.exists():
        shutil.copytree(origen, RUN_DIR_DRIVE / subcarpeta)
shutil.copy2(RUN_DIR_LOCAL / 'manifest_ejecucion.json', RUN_DIR_DRIVE / 'manifest_ejecucion.json')

print('CONTROLES APROBADOS')
print('Evidencia privada guardada en:', RUN_DIR_DRIVE)